# Regularization, cell by cell

Goodfellow et al., [ch. 7](https://www.deeplearningbook.org/contents/regularization.html). Конспект: `NOTES.md`.

**Регуляризация** = любое изменение алгоритма, которое режет **ошибку на тесте, не на трейне**. Не «сделай сеть меньше». Наоборот: огромный класс гипотез + ты делаешь дешёвыми только те решения, которым веришь.

$$
\tilde{J}(\theta; X,y) = J(\theta; X,y) + \alpha\,\Omega(\theta)
$$

$J$ — данные. $\Omega$ — prior. $\alpha$ — насколько prior орёт громче данных. $\alpha=0$ — чистый эмпирик. $\alpha\to\infty$ — игнорируешь выборку.

## Как идти

1. **Editor Window**, не Agents. Kernel Python 3 (`pip install -r requirements.txt`, нужен `ipywidgets`).
2. Setup один раз. **Shift+Enter** = ячейка + следующая.
3. Геометрия / линейные модели: **слайдеры**, тащи сразу. Fashion-ячейки учат сеть — поменял крутилку, **Ctrl+Enter**.
4. Не жми Run All в первый раз.

| § | принцип | что крутить |
|---|---|---|
| 2 | L2 / Fig 7.1 | $\alpha$ |
| 3 | L1 vs L2, когда вес умирает | $\alpha$, $H$ |
| 4 | штраф = ограничение | — |
| 5–6 | ridge / lasso на реальных фичах | ridge $\alpha$, lasso $\alpha$, $C$ |
| 8–14 | MLP: none, WD, L1, dropout, noise, smooth, early stop | `WD`, `L1`, `DROPOUT`, … |
| 15–16 | sharing + aug | `MAX_SHIFT` |
| 17–19 | sparse $h$, bagging, FGSM | `ACT_L1`, `N_BAG`, `ADV_EPS` |


## 0. Setup

Один раз за kernel. Потом прыгай по секциям. `EPOCHS` / `N_TRAIN` внизу ячейки — бюджет Fashion-сеток. Геометрия от них не зависит.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import os, sys
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / "regularization" / "train.py").exists():
    HERE = HERE / "regularization"
elif not (HERE / "train.py").exists():
    raise FileNotFoundError(f"can't find train.py from {Path.cwd()}")
os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("cwd:", HERE)

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge

from closed_form import l1_soft_threshold, l2_shrink_diag, l2_shrink_eigen, ridge_normal_equation
from data import (
    FASHION_LABELS,
    load_cancer_split,
    load_diabetes_split,
    load_fashion_split,
    shift_images,
)
from models import MLP, TinyCNN, fgsm, n_params, weight_l1
from train import (
    TrainConfig,
    bagged_proba,
    bootstrap_indices,
    metrics_for,
    train_classifier,
)
import viz
print("sliders:", "on" if viz.HAS_WIDGETS else "OFF — pip install ipywidgets, restart kernel")
print("Shift+Enter runs a cell. Drag α sliders on geometry cells; Ctrl+Enter re-runs training knobs.")

plt.rcParams.update({"figure.figsize": (6.2, 4.0), "figure.dpi": 110})

EPOCHS = 8
N_TRAIN = 800
N_VAL = 800
N_TEST = 1500
LR = 3e-3
HIDDEN = (256, 256)
SEED = 0

BOARD: dict[str, dict] = {}
MODELS: dict[str, torch.nn.Module] = {}
HISTS: dict = {}


def show_board() -> None:
    viz.scoreboard(BOARD)


def param_l2(model: torch.nn.Module) -> float:
    tot = 0.0
    for p in model.parameters():
        if p.ndim > 1:
            tot += float(p.detach().pow(2).sum())
    return tot ** 0.5


def plot_hist(hist, title: str = "") -> None:
    fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
    ax[0].plot(hist.train_acc, label="train")
    ax[0].plot(hist.val_acc, label="val")
    ax[0].set_title("accuracy")
    ax[0].set_xlabel("epoch")
    ax[0].legend()
    ax[1].plot(hist.train_loss, label="train")
    ax[1].plot(hist.val_loss, label="val")
    if getattr(hist, "best_epoch", None) is not None:
        ax[1].axvline(hist.best_epoch, ls=":", color="k", lw=1)
    ax[1].set_title("loss")
    ax[1].set_xlabel("epoch")
    ax[1].legend()
    fig.suptitle(title, y=1.02)
    plt.show()


def fit_mlp(name: str, dropout: float = 0.0, **cfg) -> tuple:
    data = DATA
    torch.manual_seed(cfg.get("seed", SEED))
    model = MLP(data.n_features, data.n_classes, hidden=HIDDEN, dropout=dropout)
    tc = TrainConfig(
        epochs=cfg.get("epochs", EPOCHS),
        lr=cfg.get("lr", LR),
        batch_size=64,
        optimizer="adamw",
        weight_decay=cfg.get("weight_decay", 0.0),
        l1=cfg.get("l1", 0.0),
        activation_l1=cfg.get("activation_l1", 0.0),
        input_noise=cfg.get("input_noise", 0.0),
        label_smoothing=cfg.get("label_smoothing", 0.0),
        early_stop_patience=cfg.get("early_stop_patience"),
        early_stop_min_epoch=cfg.get("early_stop_min_epoch", 4),
        restore_best=cfg.get("restore_best", False),
        adversarial_eps=cfg.get("adversarial_eps", 0.0),
        seed=cfg.get("seed", SEED),
    )
    hist = train_classifier(model, data.X_train, data.y_train, data.X_val, data.y_val, tc)
    met = metrics_for(
        model, data.X_train, data.y_train, data.X_val, data.y_val, data.X_test, data.y_test
    )
    BOARD[name] = {
        "train": met.train_acc,
        "val": met.val_acc,
        "test": met.test_acc,
        "gap": met.gap,
    }
    MODELS[name] = model
    HISTS[name] = hist
    print(
        f"{name:18} train={met.train_acc:.3f}  val={met.val_acc:.3f}  "
        f"test={met.test_acc:.3f}  gap={met.gap:.3f}  stopped@{hist.stopped_epoch}"
        f"  ||W||_2={param_l2(model):.1f}"
    )
    plot_hist(hist, name)
    return model, hist, met


## 1. Зачем это вообще

Интервью-строка: *ёмкость — не «меньше параметров», а «какие решения в огромном классе ты сделал дешёвыми».*

Штрафуем **веса, не байасы**. Байас — одно число на юнит: сжать его = сдвинуть порог для всех входов. Вариации почти не снимает, недообучение даёт.

Дальше каждая секция: сначала смысл и «что будет, если крутить», сразу под ней — интерактив.


## 2. L2 / ridge / weight decay — Fig 7.1

$\Omega = \tfrac12\|w\|_2^2$. Градиент штрафа $=w$: пропорционален весу, **в нуле сам выключается**. Поэтому L2 никогда не делает точный ноль (если $w^{*}\neq 0$).

Вокруг нерегуляризованного минимума $w^{*}$, гессиан $H=\nabla^2 J$:

$$
\tilde{w} = (H+\alpha I)^{-1} H w^{*}
= Q(\Lambda+\alpha I)^{-1}\Lambda Q^{\top} w^{*}
$$

В базисе собственных векторов $H$ каждая ось просто масштабируется:

$$
\tilde{w}_{i} = \frac{\lambda_{i}}{\lambda_{i}+\alpha}\,w^{*}_{i}
$$

$\lambda$ (кривизна $J$ вдоль оси) = насколько **данные** держат этот вес.

| $\lambda$ vs $\alpha$ | смысл | что видишь |
|---|---|---|
| $\lambda\gg\alpha$ | данные уверены | $\tilde{w}\approx w^{*}$, почти не едет |
| $\lambda\ll\alpha$ | ось плохо определена, высокая дисперсия | сминается к 0 |

**Слайдер $\alpha$:** тащи вверх — квадратик $\tilde{w}$ ползёт к нулю **вдоль $w_1$** (тут $\lambda=0.15$), а $w_2$ ($\lambda=2.4$) почти стоит. Синие эллипсы — уровни $J$. Красный пунктир — L2-шары.

SGD-вид (eq 7.5): каждый шаг сначала умножает $w\leftarrow(1-\epsilon\alpha)w$, потом вычитает $\epsilon\nabla J$. До нуля за конечное время не доезжает. MAP: гауссов prior на $w$.


In [ ]:
viz.play(
    viz.l2_geometry,
    alpha=viz.fslider(0.55, min=0.01, max=4.0, step=0.01, description="α (L2)"),
)


Синий кружок = $w^{*}$ (без штрафа). Красный квадрат = $\tilde{w}$. Стрелка почти горизонтальная = штраф убил слабую ось, сильную не тронул. Это и есть «L2 режет дисперсию в плохо определённых направлениях».


## 3. L1 vs L2 — в какой момент вес умирает

L1: $\Omega=\|w\|_1$. Градиент $=\alpha\,\mathrm{sign}(w)$ — **константа**, не затухает при $w\to 0$.

На диагональном $H$ (eq 7.23):

$$
\tilde{w}^{\mathrm{L1}}
= \mathrm{sign}(w^{*})\max\bigl(|w^{*}|-\alpha/H,\,0\bigr)
\qquad
\tilde{w}^{\mathrm{L2}}
= \frac{H}{H+\alpha}\,w^{*}
$$

Силы в оптимуме (1-D): данные — пружина жёсткости $H$ к $w^{*}$. L2 — вторая пружина к 0 (сила $\propto\tilde{w}$, в нуле выключается). L1 — постоянная сила $\alpha$. Если сдвиг $\alpha/H$ больше $|w^{*}|$, пружина данных не уравновешивает L1 → единственный оптимум ровно $0$.

Порог: L1 умирает при $\alpha = |w^{*}|\cdot H$, **не** при $\alpha=|w^{*}|$.

**Два слайдера, один и тот же $\alpha$ на оба штрафа.** Четыре пробы $w^{*}\in\{0.3,0.8,1.5,2.5\}$.

| крутишь | что происходит |
|---|---|
| $\alpha$ ↑ | оранжевая зона шире; в таблице `YES`, когда $\alpha\ge |w^{*}|H$; L2 только ползёт к оси |
| $H$ ↑ | данные держат вес крепче; тот же $\alpha$ уже не убивает; зона `α/H` уже |
| $H$ ↓ | слабо определённый вес (как $w_1$ в Fig 7.1) дохнет раньше |

Левый график: $\times$ = L1, квадрат = L2. Правый: путь vs $\alpha$ — solid садится на 0 и лежит, dashed (L2) только асимптота. Вертикаль = текущий $\alpha$.

Протокол: $\alpha=0.29$ — все живы. Cross $0.30$ — первый труп. Потом $0.8$, $1.5$, $2.5$. Потом зафиксируй $\alpha=0.8$ и гоняй $H$: при $H=2$ вес $0.8$ снова жив ($\alpha_{\mathrm{zero}}=1.6$).


In [ ]:
viz.play(
    viz.zero_moment,
    alpha=viz.fslider(0.5, min=0.0, max=4.0, step=0.02, description="α  L1+L2"),
    H=viz.fslider(1.0, min=0.15, max=3.0, step=0.05, description="H  кривизна J"),
)


## 4. Штраф = ограничение (ch. 7.2)

$$
\min J+\alpha\Omega
\quad\Longleftrightarrow\quad
\min J \text{ при } \Omega(\theta)\le k
$$

$\alpha$ — множитель Лагранжа для какого-то $k(\alpha)$. Больше $\alpha$ ⇔ меньше допустимый шар/ромб.

Красные маркеры: L2-диск — касание эллипса $J$ **не на оси** (обе координаты живы). L1-ромб — касание **в вершине** → эта координата ровно 0. Это та же геометрия, что soft-threshold в §3.

Hard constraint (проецируй $\|w\|$ обратно на шар) vs penalty: проекция не даёт весам взорваться численно. Штраф мягче и удобнее в SGD.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.0), layout="constrained")
axes[0].plot(np.cos(theta), np.sin(theta), color="#1f77b4", lw=2)
axes[0].plot([0.72], [0.69], "o", color="#d62728", ms=9)
axes[0].annotate("off-axis\n(both coords live)", xy=(0.72, 0.69), xytext=(-0.2, 1.05),
                 fontsize=8, arrowprops=dict(arrowstyle="->", color="#444"))
axes[0].set_title(r"$L_2$ ball $\|w\|_2 \leq k$")
diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]], float)
axes[1].plot(diamond[:, 0], diamond[:, 1], color="#ff7f0e", lw=2)
axes[1].plot([0.0], [1.0], "x", color="#d62728", ms=12, mew=2)
axes[1].annotate("$w_1 = 0$", xy=(0.0, 1.0), xytext=(0.45, 0.55),
                 fontsize=9, color="#d62728", arrowprops=dict(arrowstyle="->", color="#d62728"))
axes[1].set_title(r"$L_1$ diamond $\|w\|_1 \leq k$")
w1 = np.linspace(-1.4, 1.4, 200)
W1, W2 = np.meshgrid(w1, w1)
J = 0.5 * ((W1 - 1.15) ** 2 / 0.7 + (W2 - 1.05) ** 2 / 0.55)
for ax in axes:
    ax.contour(W1, W2, J, levels=6, colors="#888", linewidths=0.8)
    ax.axhline(0, color="#ccc", lw=0.6)
    ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel(r"$w_1$")
    ax.set_ylabel(r"$w_2$")
plt.show()


## 5. Линейка: diabetes — lasso реально зануляет

Те же L2/L1, но на 10 реальных фичах (Efron/Hastie), $n_{\mathrm{train}}=80$, фичи уже scaled.

| имя | что |
|---|---|
| `age`, `sex`, `bmi`, `bp` | возраст, пол, ИМТ, давление |
| `s1` | tc, общий холестерин |
| `s2` | ldl |
| `s3` | hdl |
| `s4` | tch = total chol / HDL |
| `s5` | ltg, (log) триглицериды |
| `s6` | glu, сахар |

`s1`–`s4` — один липидный кластер, коллинеарны. OLS/ridge размазывают кредит по кластеру (мелкий шум на каждом). L1 оставляет сильного (`bmi` / `s5`) и сажает остальных в 0.

| | лосс | нули? |
|---|---|---|
| OLS | $\|Xw-y\|_2^2$ | нет |
| ridge | ${}+ \alpha\|w\|_2^2$ | нет, только сжимает |
| lasso | ${}+ \alpha\|w\|_1$ | да, красные $\times$ |

**Слайдеры:** `ridge α` и `lasso α` — **разные шкалы**, не один и тот же $\alpha$ из §3.

| крутишь | ожидай |
|---|---|
| lasso $\alpha$ ↑ | больше красных $\times$, test MSE сначала вниз (выкинул шум), потом вверх (выкинул сигнал) |
| lasso $\alpha\to 0$ | как OLS, нулей нет |
| ridge $\alpha$ ↑ | стебли короче, $\times$ не появятся |

Дефолт $0.8$ обычно убивает `s1` и `s4`.


In [ ]:
d = load_diabetes_split()  # used by the closed-form cell below
viz.play(
    viz.diabetes_stems,
    ridge_alpha=viz.fslider(2.0, min=0.0, max=20.0, step=0.1, description="ridge α", continuous_update=False),
    lasso_alpha=viz.fslider(0.8, min=0.01, max=8.0, step=0.01, description="lasso α", continuous_update=False),
)


Тот же lasso, полный path по $\alpha$ (лог-ось). Слева каждая фича — линия: падает на ось и лежит. Справа: число нулей растёт, test MSE — яма, потом рост. Яма = profitable bias. Это и имеют в виду под «L1 делает feature selection».


In [ ]:
viz.lasso_zero_path(d.X_train, d.y_train, d.X_test, d.y_test, d.feature_names)


Проверка, что $\alpha=0$ в нормальном уравнении совпадает со sklearn OLS (косинус $\approx 1$). К L1 отношения не имеет — якорь, что closed form не сломан.


In [ ]:
d = load_diabetes_split()
ols = LinearRegression().fit(d.X_train, d.y_train)
Xb = np.c_[d.X_train, np.ones(len(d.X_train))]
closed = ridge_normal_equation(Xb, d.y_train, alpha=0.0)
align = float(
    np.dot(closed[:-1], ols.coef_)
    / (np.linalg.norm(closed[:-1]) * np.linalg.norm(ols.coef_) + 1e-12)
)
print("OLS vs closed-form cosine:", round(align, 6))


## 6. Logistic L1 на breast_cancer

30 фич, $n_{\mathrm{train}}=40$ — unregularized легко делает train $=1$ (7.3: недоопределено). L1 оставляет кучку (часто `worst concave points`).

sklearn 1.8+: не трогай `penalty`. `l1_ratio=0` → L2, `=1` → L1, `C=np.inf` → none. **C = 1/α**: меньше C = больше штраф.

| `C_L1` | |
|---|---|
| 0.2 | очень дырявый, мало фич |
| 0.8 | дефолт, ~6 ненулей |
| 5.0 | почти плотный |

Поменяй `C_L1`, Ctrl+Enter. Следующая ячейка — path: ось C инвертирована, «налево = больше L1».


In [ ]:
C_L1 = 0.8   # try 0.2 (very sparse) vs 5.0 (almost dense)
C_L2 = 0.5

c = load_cancer_split()
models = {
    "none": LogisticRegression(C=np.inf, max_iter=4000, random_state=0),
    "L2": LogisticRegression(C=C_L2, l1_ratio=0.0, max_iter=4000, random_state=0),
    "L1": LogisticRegression(C=C_L1, l1_ratio=1.0, solver="saga", max_iter=8000, random_state=0),
}
for name, m in models.items():
    m.fit(c.X_train, c.y_train)
    z = int(np.sum(np.abs(m.coef_) < 1e-4))
    print(
        f"{name:5}  train={m.score(c.X_train, c.y_train):.3f}  "
        f"test={m.score(c.X_test, c.y_test):.3f}  zeros={z}/{m.coef_.size}"
    )

w = models["L1"].coef_.ravel()
print("\nL1 survivors:")
for i in np.argsort(-np.abs(w)):
    if abs(w[i]) < 1e-4:
        continue
    print(f"  {c.feature_names[i]:24}  {w[i]:+.3f}")

viz.stems_with_zeros(
    {"logreg L1": w},
    c.feature_names,
    title="breast_cancer L1 — red × dropped features",
)


In [ ]:
viz.logreg_zero_path(c.X_train, c.y_train, c.feature_names)


## 7. Fashion-MNIST

28×28 одежда, 10 классов. Первый запуск качает ~30MB в `data_cache/`. Дальше все MLP/CNN на этом сплите. Маленький $n$ (дефолт 800) специально, чтобы жирная 256-256 могла оверфитить — иначе regularizer нечему лечить.


In [ ]:
DATA = load_fashion_split(n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED)
print("train", DATA.X_train.shape, "val", DATA.X_val.shape, "test", DATA.X_test.shape)

n, side = 40, 28
fig, axes = plt.subplots(4, 10, figsize=(11, 4.4), layout="constrained")
for ax, img, lab in zip(axes.ravel(), DATA.X_train[:n], DATA.y_train[:n]):
    ax.imshow(img.reshape(side, side), cmap="gray")
    ax.set_title(FASHION_LABELS[int(lab)], fontsize=7)
    ax.axis("off")
fig.suptitle("Fashion-MNIST train subset")
plt.show()


## 8. Unregularized MLP — больной, которого лечим

Жирная 256-256, маленький $n$, без штрафа. Это **оверфит-бейзлайн**. Потом сравниваешь через `BOARD` / `MODELS["none"]`.

Жди: train ≫ test, `gap` большой, первый слой плотный. Кривые: train лезет вверх, val отстаёт. `restore_best=False` — оставляем **последнюю** эпоху, не лучшую.


In [ ]:
model_none, hist_none, met_none = fit_mlp("none", restore_best=False, seed=0)
viz.first_layer_abs(MODELS["none"], title="none: first layer is dense")


## 9. L2 weight decay на сети (ch. 7.1)

То же $\Omega=\tfrac12\|w\|_2^2$, но AdamW: decay на апдейте, только тензоры `ndim>1` (байасы скип). Не путать с `Adam + L2 в лоссе` — для адаптивных методов это не то же самое.

**Смотри:** гистограмма / CDF $|w|$ vs `none`. L2 коротит хвост, **спайка в 0 нет**.

| `WD` | |
|---|---|
| `1e-3` | ≈ none |
| `0.04` | дефолт, `‖W‖₂` чуть падает, gap часто чуть меньше |
| `0.2` | underfit: train и test оба вниз |

Поменял `WD` → Ctrl+Enter. Если `none` уже гонял, CDF наложится.


In [ ]:
WD = 0.04  # try 1e-3, 0.04, 0.2
_ = fit_mlp("l2", weight_decay=WD, seed=1)
if "none" in MODELS:
    print(f"||W||_2 ratio l2/none = {param_l2(MODELS['l2']) / param_l2(MODELS['none']):.3f}")
    viz.weight_sparsity({"none": MODELS["none"], "l2": MODELS["l2"]}, title="L2 shrinks |w|, no extra zeros")
show_board()


## 10. L1 на веса сети — дырки в $W$

В лосс: `+= L1 * Σ|W|` (не байасы). Тот же принцип, что lasso, но Adam не даёт машинный ноль как coordinate descent: «ноль» = куча массы у мелкого $|w|$.

**Смотри:** CDF $|w|$ — L1 **прыгает** у малого порога; справа heatmap первого слоя, белое = $|W|<10^{-3}$.

| `L1` | |
|---|---|
| `1e-4` | почти как none |
| `8e-4` | дефолт, дырки видны если эпох хватило |
| `3e-3` | дырявее, test может просесть |

Сравни `frac<1e-3` у `none` / `l2` / `l1`.


In [ ]:
L1 = 8e-4  # try 1e-4, 8e-4, 3e-3
_ = fit_mlp("l1", l1=L1, seed=2)
print("weight L1", float(weight_l1(MODELS["l1"]).detach()))
cmp = {k: MODELS[k] for k in ("none", "l2", "l1") if k in MODELS}
viz.weight_sparsity(cmp, title="L1 vs L2 vs none — L1 CDF jumps")
viz.first_layer_abs(MODELS["l1"], title="L1: white pixels are near-zero weights")
show_board()


## 11. Dropout (ch. 7.12) — тоже 7.5

На трейне каждый скрытый юнит $\times\mathrm{Bernoulli}(1-p)$. PyTorch — **inverted**: на трейне уже делит на $1-p$, на eval маски нет. Иначе на тесте активации были бы в $(1-p)$ раз меньше.

Три интерпретации сразу:

1. мультипликативный шум на $h$ / аугмент на каждом слое;
2. дешёвый ансамбль $2^{n}$ подсетей с общими весами;
3. нельзя коадаптироваться — партнёр может не прийти.

**Смотри heatmap:** строки = разные train-форварды **одной** картинки. Чёрное мерцает (дроп). Нижний ряд eval — плотный, это среднее по маскам. Train acc часто < val: на трейне шум, это не баг.

| `DROPOUT` $p$ | |
|---|---|
| 0.2 | слабо (книга — для входа) |
| 0.5 | книга для hidden, дефолт |
| 0.8 | почти пустая сеть на каждом форварде, underfit, heatmap чёрный |


In [ ]:
DROPOUT = 0.5  # try 0.2, 0.5, 0.8
_ = fit_mlp("dropout", dropout=DROPOUT, seed=3)
viz.dropout_passes(MODELS["dropout"], DATA.X_test, title=f"dropout p={DROPOUT}: mask flickers at train, dense at eval")
show_board()


## 12. Входной шум (ch. 7.5)

`x ← x + σ N(0,1)` только на трейне. Бесконечно малый $\sigma$ ≈ weight decay (Bishop). Конечный $\sigma$ **строго сильнее** L2: это не квадратичный штраф на $w$, а штраф на чувствительность ко всем направлениям входа. Не путать с FGSM: здесь шум изотропный, там — наихудший.

**Смотри сетку:** низ — то, на чём учишься.

| `NOISE` | |
|---|---|
| 0.05 | лёгкий blur ≈ мелкий WD |
| 0.15 | дефолт, уже видно |
| 0.4 | картинка умерла, underfit |


In [ ]:
NOISE = 0.15  # try 0.05, 0.15, 0.4
viz.noise_grid(DATA.X_train, NOISE, n=8)
_ = fit_mlp("input_noise", input_noise=NOISE, seed=4)
show_board()


## 13. Label smoothing (ch. 7.5.1)

One-hot $y$ заменяем на $(1-\epsilon)y + \epsilon/K$. Считаем, что метка врёт с вероятностью $\epsilon$. Сеть больше не должна загонять логиты в $\pm\infty$ (обычный CE у one-hot этого требует).

**Смотри гистограмму max-softmax** на тесте vs `none`: меньше массы у 1.0.

| `EPS` | |
|---|---|
| 0.05 | чуть скромнее |
| 0.1 | дефолт / книга |
| 0.3 | уже сильно врём себе в метку, калибровка плывёт |


In [ ]:
EPS = 0.1  # try 0.05, 0.1, 0.3
_ = fit_mlp("label_smooth", label_smoothing=EPS, seed=6)
cmp = {k: MODELS[k] for k in ("none", "label_smooth") if k in MODELS}
viz.confidence_hist(cmp, DATA.X_test[:800])
show_board()


## 14. Early stopping (ch. 7.8)

Самый дешёвый регуляризатор в DL. Время обучения = ручка ёмкости. Val растёт, потом падает (запомнил трейн). Берём снимок **лучшего val**, не последний $\theta$.

При квадратичном $J$ + GD это **то же самое, что L2** (Bishop / Sjöberg–Ljung): число шагов $\leftrightarrow 1/\alpha$. На практике выгоднее L2-поиска: один прогон. Цена — нужен val и хранить snapshot.

`none` оставляет последнюю эпоху. Здесь `restore_best=True`, вертикаль = `best_epoch`.

| крутилка | |
|---|---|
| `PATIENCE` 2 | стоп рано, риск недоучить |
| `PATIENCE` 4 | дефолт |
| `PATIENCE` 15 | почти как none, но откат к пику val всё равно |
| `MIN_EPOCH` | patience не считается до этой эпохи — иначе везучий val на эпохе 1 убивает ран |


In [ ]:
PATIENCE = 4
MIN_EPOCH = 4
_ = fit_mlp(
    "early_stop",
    epochs=max(EPOCHS, 16),
    early_stop_patience=PATIENCE,
    early_stop_min_epoch=MIN_EPOCH,
    restore_best=True,
    seed=5,
)
print("best_epoch", HISTS["early_stop"].best_epoch, "stopped", HISTS["early_stop"].stopped_epoch)
ov = {k: HISTS[k] for k in ("none", "early_stop") if k in HISTS}
viz.overlay_val(ov, title="early stop restores the red dot, not the last epoch")
show_board()


Стек dropout + L2, если не лень. Два prior сразу: маленькие веса **и** «не коадаптируйся». Не обязательно лучше каждого по отдельности — это просто ещё одна точка на scoreboard.


In [ ]:
_ = fit_mlp("l2+dropout", dropout=0.4, weight_decay=0.02, seed=7)
show_board()


## 15. Sharing vs tying (ch. 7.9)

- **Tying:** штраф $\|w_A-w_B\|^2$ — веса похожи, но не обязаны совпадать.
- **Sharing:** один и тот же тензор. CNN: один $3\times 3$ на всех позициях = prior трансляции + на порядок меньше уникальных чисел.

Слайдера нет: либо архитектура такая, либо нет. Сравни `n_params` и test у TinyCNN vs MLP 256-256 на том же сплите. Картинка 16 ядер — это и есть расшаренные веса.

«Больше параметров ⇒ больше оверфит» здесь ломается: 20k CNN часто бьёт 269k MLP.


In [ ]:
IMAGES = load_fashion_split(
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED, as_images=True
)
cnn = TinyCNN(img_size=28)
viz.param_bars({"MLP 256-256": n_params(MLP(784, 10, hidden=HIDDEN)), "TinyCNN": n_params(cnn)})
hist_cnn = train_classifier(
    cnn,
    IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=11),
)
met_cnn = metrics_for(
    cnn, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn"] = {"train": met_cnn.train_acc, "val": met_cnn.val_acc, "test": met_cnn.test_acc, "gap": met_cnn.gap}
MODELS["cnn"] = cnn
HISTS["cnn"] = hist_cnn
print(f"cnn  train={met_cnn.train_acc:.3f}  test={met_cnn.test_acc:.3f}  gap={met_cnn.gap:.3f}")
plot_hist(hist_cnn, "cnn")
viz.conv_kernels(cnn)
show_board()


## 16. Аугментация (ch. 7.4) = монте-карло 7.14

Делаешь новые $(x,y)$, меняя $x$ **без смены метки**. Prior: «сдвиг на 2 пикселя — всё ещё та же футболка». Tangent prop (7.14, не кодировали) штрафует $\|\nabla_x f\cdot v\|$ аналитически; сдвиги — стохастическая версия.

Не флипать: 6/9, рубашка vs нет.

**Смотри сетку:** низ — то, что дописывается в трейн.

| крутилка | |
|---|---|
| `MAX_SHIFT` 1 | почти незаметно |
| `MAX_SHIFT` 2 | дефолт |
| `MAX_SHIFT` 4 | уже каша, можно убить метку семантикой |
| `N_COPIES` ↑ | больше фейковых точек, эпоха дольше, gap обычно меньше |


In [ ]:
MAX_SHIFT = 2     # try 1, 2, 4
N_COPIES = 2

shifted = shift_images(IMAGES.X_train, max_shift=MAX_SHIFT, seed=20)
viz.shift_grid(IMAGES.X_train, shifted, n=8)

aug_X = [IMAGES.X_train]
aug_y = [IMAGES.y_train]
for i in range(N_COPIES):
    aug_X.append(shift_images(IMAGES.X_train, max_shift=MAX_SHIFT, seed=20 + i))
    aug_y.append(IMAGES.y_train)
X_aug, y_aug = np.concatenate(aug_X), np.concatenate(aug_y)
print("train size", len(IMAGES.y_train), "→", len(y_aug))

cnn_aug = TinyCNN(img_size=28)
hist_aug = train_classifier(
    cnn_aug, X_aug, y_aug, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=12),
)
met_aug = metrics_for(
    cnn_aug, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn+aug"] = {"train": met_aug.train_acc, "val": met_aug.val_acc, "test": met_aug.test_acc, "gap": met_aug.gap}
MODELS["cnn+aug"] = cnn_aug
print(f"cnn+aug  train={met_aug.train_acc:.3f}  test={met_aug.test_acc:.3f}  gap={met_aug.gap:.3f}")
plot_hist(hist_aug, "cnn+aug")
show_board()


## 17. Sparse representations (ch. 7.10)

Это **не** lasso на $W$. Две разные редкости:

| | что ноль | штраф | смысл |
|---|---|---|---|
| sparse **parameters** | веса $W$ | L1 на $W$ (§10) | выкинуть фичу навсегда |
| sparse **representation** | код $h=f(x)$ | L1 на $h$ | этот $x$ объясняется немногими факторами |

Плотный $W$ спокойно даёт дырявый $h$. В лоссе: `+ ACT_L1 * mean(|h|)`. Градиент по $h$ тянет активации к 0, $W$ меняется только косвенно.

Почему **tanh, не ReLU:** ReLU и так нули — эффекта не видно. Tanh без штрафа сидит у $\pm 1$.

**Смотри 3 панели:** гистограмма $h$ (пик в 0 vs горбы у $\pm 1$); heatmap sample×unit — яркая каша vs тёмные дырки.

| `ACT_L1` | |
|---|---|
| 0 | горбы $\pm 1$, heatmap яркий |
| 0.15 | пик в 0, темнее; на 8 эпохах сдвиг мелкий |
| 0.4 | почти всё $h\approx 0$, сетка тупая |

Юнит может гореть на одних картинках и быть мёртвым на других — это не выкинутая фича.


In [ ]:
ACT_L1 = 0.15  # try 0, 0.05, 0.15, 0.4

def hidden_of(act_l1: float, name: str):
    torch.manual_seed(30)
    model = MLP(DATA.n_features, DATA.n_classes, hidden=(64, 64), activation="tanh")
    train_classifier(
        model, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            activation_l1=act_l1, seed=30,
        ),
    )
    model.eval()
    with torch.no_grad():
        _, h = model(torch.from_numpy(DATA.X_test[:200]), return_hidden=True)
    h = h.numpy()
    print(f"{name:18} mean|h|={np.mean(np.abs(h)):.3f}  frac≈0={np.mean(np.abs(h)<1e-2):.3f}")
    return h

h0 = hidden_of(0.0, "tanh")
h1 = hidden_of(ACT_L1, "tanh + L1 on h")
viz.hidden_sparsity(h0, h1, "tanh", f"L1={ACT_L1} on h")


## 18. Bagging (ch. 7.11)

$k$ моделей на бутстрэп-выборках, усреднение softmax. Каждый оверфитит свой шум, среднее гасит некоррелированные ошибки. Dropout — дешёвый **неявный** bag из $2^n$ подсетей с общими весами. Здесь ансамбль явный, веса не шарятся.

**Смотри:** столбики мемберов vs bag. Bag должен бить типичного мембера.

| `N_BAG` | |
|---|---|
| 2 | уже лучше среднего мембера, шумно |
| 3 | дефолт |
| 5 | лучше, цена = $k$ полных трейнов, убывающая отдача |


In [ ]:
N_BAG = 3  # try 2, 3, 5

members = []
member_test = []
for i in range(N_BAG):
    idx = bootstrap_indices(len(DATA.X_train), seed=20 + i)
    m = MLP(DATA.n_features, DATA.n_classes, hidden=HIDDEN)
    train_classifier(
        m,
        DATA.X_train[idx], DATA.y_train[idx], DATA.X_val, DATA.y_val,
        TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=20 + i),
    )
    members.append(m)
    te = metrics_for(m, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val, DATA.X_test, DATA.y_test).test_acc
    member_test.append(te)
    print(f"  member {i} test={te:.3f}")

bag_tr = float(np.mean(bagged_proba(members, DATA.X_train).argmax(1) == DATA.y_train))
bag_va = float(np.mean(bagged_proba(members, DATA.X_val).argmax(1) == DATA.y_val))
bag_te = float(np.mean(bagged_proba(members, DATA.X_test).argmax(1) == DATA.y_test))
BOARD[f"bag x{N_BAG}"] = {"train": bag_tr, "val": bag_va, "test": bag_te, "gap": bag_tr - bag_te}
print(f"{f'bag x{N_BAG}':18} train={bag_tr:.3f}  val={bag_va:.3f}  test={bag_te:.3f}  gap={bag_tr-bag_te:.3f}")
viz.bag_bars(member_test, bag_te, bag_tr)
show_board()


## 19. FGSM / adversarial (ch. 7.13)

Случайный шум изотропен. Атака — **наихудший** сдвиг в $L_\infty$-шаре:

$$
x_{\mathrm{adv}} = x + \varepsilon\,\mathrm{sign}(\nabla_x J)
$$

Выглядит как шум, выровнен по градиенту. Чистая сеть на Fashion часто падает с $\sim 0.8$ до $\sim 0.15$ уже при $\varepsilon=0.12$. Учишь 50/50 clean+FGSM — поверхность локально площе, та же атака бьёт слабее, clean почти не падает.

Это другой prior, чем `NOISE`: не «устойчив к любому мелкому шуму», а «устойчив к направленному удару».

**Смотри триплеты:** clean / $\mathrm{sign}(\nabla_x J)$ / adv. Красная подпись = класс перевернулся. Потом бары clean-train vs FGSM-train.

| `ADV_EPS` | |
|---|---|
| 0.03 | еле видно, мало флипов |
| 0.12 | дефолт |
| 0.25 | mag, картинки разные; если так учить — сильнее defense, риск портить clean |


In [ ]:
ADV_EPS = 0.12  # try 0.03, 0.12, 0.25

def acc(model, X, y, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        pred = model(xt[i:i + batch]).argmax(1)
        correct += int((pred == yt[i:i + batch]).sum())
        n += pred.numel()
    return correct / n


def fgsm_acc(model, X, y, eps, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        xb, yb = xt[i:i + batch], yt[i:i + batch]
        adv = fgsm(model, xb, yb, eps)
        pred = model(adv).argmax(1)
        correct += int((pred == yb).sum())
        n += yb.numel()
    return correct / n

rows = []
trained = {}
for name, adv in [("clean train", 0.0), ("FGSM train", ADV_EPS)]:
    m = TinyCNN(img_size=28)
    train_classifier(
        m, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            adversarial_eps=adv, seed=40,
        ),
    )
    trained[name] = m
    rows.append({
        "name": name,
        "clean": acc(m, IMAGES.X_test, IMAGES.y_test),
        "fgsm": fgsm_acc(m, IMAGES.X_test, IMAGES.y_test, ADV_EPS),
    })
    print(f"{name:12}  clean={rows[-1]['clean']:.3f}  FGSM={rows[-1]['fgsm']:.3f}")

print("\nattack on the clean net (captions: true → pred, red = flipped):")
viz.fgsm_triplets(trained["clean train"], IMAGES.X_test, IMAGES.y_test, ADV_EPS, FASHION_LABELS, n=6)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
x = np.arange(len(rows))
ax.bar(x - 0.18, [r["clean"] for r in rows], 0.36, label="clean test")
ax.bar(x + 0.18, [r["fgsm"] for r in rows], 0.36, label="FGSM test", color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels([r["name"] for r in rows])
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title(rf"FGSM $\varepsilon={ADV_EPS}$")
plt.show()


## 20. Scoreboard

`gap = train − test` на чистом Fashion. Перезапусти после любой train-ячейки.

Как читать одной фразой: поднимаешь $\alpha$ / $p$ / $\varepsilon$ / стоп раньше → больше bias, меньше variance. Слишком много — оба acc падают. Слишком мало — train→1, test отстаёт.


In [ ]:
show_board()


## 21. Карта prior'ов и что ещё крутить

| prior | механизм | ручка |
|---|---|---|
| веса маленькие, ни одно направление не выделяй | L2 / WD / early stop (≈L2) | $\alpha$, `WD`, число эпох |
| мало ненулевых весов | L1 / lasso | $\alpha$, `C`↓ |
| мало активных юнитов на данный $x$ | L1 на $h$, ReLU, dropout | `ACT_L1`, $p$ |
| метка не требует бесконечных логитов | label smoothing | $\epsilon$ |
| мелкий сдвиг пикселей не меняет класс | aug / CNN sharing | `MAX_SHIFT` |
| мелкий *худший* сдвиг не меняет класс | FGSM train | `ADV_EPS` |
| ошибка одного фита случайна | bagging / dropout-ансамбль | `N_BAG`, $p$ |

Попробуй:

- lasso $\alpha$ / `C_L1`, пока красные $\times$ не появятся / не исчезнут;
- §3: $\alpha=0.8$, гоняй $H$ — вес $0.8$ оживает при $H>1$;
- `L1=3e-3` на MLP, если дырки тёмные на 8 эпохах;
- `DROPOUT=0.8` и смотри маску;
- `ADV_EPS=0.25`;
- `EPOCHS=25`, `N_TRAIN=1500` — цифры ближе к `run.py`.

7.6/7.7 (semi-sup / multitask) и 7.14 (tangent prop) в тетради нет: unlabeled $P(x)$ и штраф $\|\nabla_x f\cdot v\|$. Аугмент — монте-карло 7.14.

Интервью-вопросы — в `NOTES.md`.
